In [19]:
import pandas as pd
import pm4py

In [20]:
# Load dataset
filepath = 'data/UpFLux_Healthcare_Database_labeled.csv'
df = pd.read_csv(filepath, parse_dates=["time:timestamp"])

df['case:concept:name'] = df['case:concept:name'].astype(str)
df['time:timestamp']= pd.to_datetime(df['time:timestamp'])

df.head()

,case:concept:name,Idade,Sexo,Nível de Urgência,CID,Médico Responsável,Doença,Data Inicial,time:timestamp,concept:name,Item,Data Prescrição,Tempo Processo,Setor,Quantidade,Retorno,Multipassante,Convênio,outlier_label
0,5446538,31,F,Não informado,J111,Médico 72,J11.1 Influenza c/out manif resp dev virus n i...,21/07/2020 10:22,2020-07-21 10:22:00,Atendimento,Atendimento,NaN,NaN,Setor Pronto Atendimento,0.0,Sem retorno,Não,Convênio 9,outlier
1,5446538,31,F,Não informado,J111,Médico 72,J11.1 Influenza c/out manif resp dev virus n i...,21/07/2020 10:45,2020-07-21 10:49:00,Triagem,Triagem,NaN,NaN,Setor Pronto Atendimento,0.0,Sem retorno,NaN,Convênio 9,outlier
2,5446538,31,F,Não informado,J111,Médico 72,J11.1 Influenza c/out manif resp dev virus n i...,21/07/2020 11:01,2020-07-21 11:01:00,Exames Laboratoriais,Coronavírus COVID-19 - Diagnóstico Molecular (...,21/07/2020 10:51,9.0,Setor Pronto Atendimento,1.0,Sem retorno,NaN,Convênio 9,outlier
3,5446538,31,F,Não informado,J111,Médico 72,J11.1 Influenza c/out manif resp dev virus n i...,21/07/2020 11:32,2020-07-21 11:32:00,Consulta,Consulta,NaN,NaN,Setor Pronto Atendimento,0.0,Sem retorno,NaN,Convênio 9,outlier
4,5446538,31,F,Não informado,J111,Médico 72,J11.1 Influenza c/out manif resp dev virus n i...,21/07/2020 19:27,2020-07-21 19:27:00,Alta,Alta para completar tratamento,NaN,NaN,Setor Pronto Atendimento,0.0,Sem retorno,NaN,Convênio 9,outlier


In [21]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (3093, 19)
Columns: ['case:concept:name', 'Idade', 'Sexo', 'Nível de Urgência', 'CID', 'Médico Responsável', 'Doença', 'Data Inicial', 'time:timestamp', 'concept:name', 'Item', 'Data Prescrição', 'Tempo Processo', 'Setor', 'Quantidade', 'Retorno', 'Multipassante', 'Convênio', 'outlier_label']


In [22]:
print("Initial dataset Info:")
print("Number of events:", len(df))
print("Number of cases:", len(df['case:concept:name'].unique()))

Initial dataset Info:
Number of events: 3093
Number of cases: 443


In [23]:
print("Dataset Info:")
print(df.info())

print("\nUnique values per column:")
print(df.nunique(dropna=False))

print("\nMissing values per column:")
print(df.isna().sum())

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 3093 entries, 0 to 3092
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   case:concept:name   3093 non-null   str           
 1   Idade               3093 non-null   int64         
 2   Sexo                3093 non-null   str           
 3   Nível de Urgência   3093 non-null   str           
 4   CID                 3093 non-null   str           
 5   Médico Responsável  3093 non-null   str           
 6   Doença              3093 non-null   str           
 7   Data Inicial        3091 non-null   str           
 8   time:timestamp      3075 non-null   datetime64[us]
 9   concept:name        3093 non-null   str           
 10  Item                3093 non-null   str           
 11  Data Prescrição     378 non-null    str           
 12  Tempo Processo      377 non-null    float64       
 13  Setor               3093 non-null   str      

### Cleaning and filtering
There are some events with missing timestamp, that is a critical value.

We should drop duplicates event, rows with missing critical values and cases with duration 0.

In [24]:
# Drop duplicates and rows with missing critical values
filtered_df = df.drop_duplicates()
filtered_df = filtered_df.dropna(subset=["case:concept:name", "concept:name", "time:timestamp"])

print("After dropping duplicates and missing values:")
print("Number of events:", len(filtered_df))
print("Number of cases:", len(filtered_df["case:concept:name"].unique()))

After dropping duplicates and missing values:
Number of events: 3071
Number of cases: 443


In [25]:
# Sort the dataframe by case and timestamp to ensure correct order of events
filtered_df = filtered_df.sort_values(by=["case:concept:name", "time:timestamp"])

case_durations = filtered_df.groupby("case:concept:name")[("time:timestamp")].agg(["min", "max"])
case_durations["duration_seconds"] = (case_durations["max"] - case_durations["min"]).dt.total_seconds()

# Filter out cases with non-positive durations
valid_cases = case_durations[case_durations["duration_seconds"] > 0].index
filtered_df = filtered_df[filtered_df["case:concept:name"].isin(valid_cases)]

print("After filtering out cases with non-positive durations:")
print("Number of events:", len(filtered_df))
print("Number of cases:", len(filtered_df["case:concept:name"].unique()))

After filtering out cases with non-positive durations:
Number of events: 3071
Number of cases: 443


In [26]:
# Select relevant columns for the event log
relevant_df = filtered_df[["case:concept:name", "concept:name", "time:timestamp", "Doença", "Retorno", "Item", "Médico Responsável", "outlier_label"]].copy()

pm4py_df = pm4py.format_dataframe(relevant_df)
log = pm4py.convert_to_event_log(pm4py_df)

df_clean = pm4py.convert_to_dataframe(log)

In [27]:
# Save the cleaned dataset to a new CSV file
filepath_clean = "data/preprocessed_dataset.csv"

df_clean.to_csv(filepath_clean, index=False)
print(f"Data cleaning and filtering completed. Cleaned dataset saved as '{filepath_clean}'.")

Data cleaning and filtering completed. Cleaned dataset saved as 'data/preprocessed_dataset.csv'.
